# Module 7: Sampling & Inference

Sampling is where everything pays off. We start from **pure noise** — random static with no structure at all — and gradually reveal a coherent image, one denoising step at a time.

Here's what we'll work through together:

- **DDPM** reverse process (the original, Algorithm 2 from Ho et al. 2020)
- **DDIM** sampling with tunable stochasticity via $\eta$
- **Accelerated sampling** using fewer timesteps
- The **probability flow ODE** connecting diffusion to continuous dynamics

By the end, you'll have a complete sampling pipeline that turns Gaussian noise into MNIST digits.

**Estimated time:** 2–3 hours

**Key references:**
- [DDPM — Ho et al. 2020](https://arxiv.org/abs/2006.11239) (Algorithm 2)
- [DDIM — Song et al. 2020](https://arxiv.org/abs/2010.02502)
- [Progressive Distillation — Salimans & Ho 2022](https://arxiv.org/abs/2202.00512)

In [ ]:
import sys
import os
import time
import math
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Shared utilities
sys.path.insert(0, '.')
from utils.schedule import cosine_schedule, get_schedule
from utils.visualization import show_images, denormalize, show_denoising_trajectory, set_style
from utils.data import get_mnist_dataloader, get_device
from utils.diffusion import prepare_schedule

# Reproducibility
torch.manual_seed(42)

# Device setup
device = get_device()
print(f"Using device: {device}")

set_style()

## Model Definition and Checkpoint Loading

Let's load the UNet we trained in Module 6. We're using the same architecture from `utils/unet.py`, so the checkpoint loads directly — no compatibility issues.

If no checkpoint is found, we'll train a quick fallback model so the rest of the notebook still works.

In [ ]:
# Import the canonical UNet (same architecture used in Modules 4 and 6)
from utils.unet import UNet

print(f"UNet imported from utils.unet")

In [ ]:
# Load checkpoint or train a quick fallback model

T = 1000
schedule = prepare_schedule(get_schedule("cosine", T=T), device)

model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4)).to(device)

checkpoint_path = "checkpoints/ddpm_mnist.pt"
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}")
    state = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(state, dict) and 'ema_shadow' in state:
        # Load EMA weights if available (better quality)
        ema_shadow = state['ema_shadow']
        for name, param in model.named_parameters():
            if name in ema_shadow:
                param.data.copy_(ema_shadow[name])
        print("Loaded EMA weights from checkpoint.")
    elif isinstance(state, dict) and 'model_state_dict' in state:
        model.load_state_dict(state['model_state_dict'])
        print("Loaded model weights from checkpoint.")
    else:
        model.load_state_dict(state)
        print("Loaded raw state dict.")
else:
    print(f"No checkpoint found at {checkpoint_path}. Training a quick model...")
    from utils.diffusion import train_step as _train_step

    dataloader = get_mnist_dataloader(batch_size=64, image_size=28)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

    model.train()
    step = 0
    max_steps = 2000
    losses = []

    while step < max_steps:
        for batch, _ in dataloader:
            if step >= max_steps:
                break
            batch = batch.to(device)  # (B, 1, 28, 28)
            loss_val = _train_step(model, batch, optimizer, schedule)
            losses.append(loss_val)
            step += 1
            if step % 500 == 0:
                print(f"  Step {step}/{max_steps}, Loss: {loss_val:.4f}")

    print(f"Training complete. Final loss: {losses[-1]:.4f}")

model.eval()
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

## DDPM Sampling (The Reverse Process)

During training, we learned to predict the noise $\epsilon_\theta(x_t, t)$ that was added to a clean image. Now we **reverse** the process: start from pure noise $x_T \sim \mathcal{N}(0, I)$ and iteratively denoise to reveal an image.

The intuition: at each step, the model looks at the current noisy image and estimates what noise is present. We subtract (a scaled version of) that noise, then optionally add a small amount of fresh noise to keep the process stochastic.

**The update rule** (Algorithm 2 from Ho et al. 2020):

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t z$$

where $z \sim \mathcal{N}(0, I)$ for $t > 1$ and $z = 0$ for $t = 1$.

### What is $\sigma_t$?

There are two common choices:

- $\sigma_t^2 = \beta_t$ — the simpler choice from Ho et al.
- $\sigma_t^2 = \tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t} \beta_t$ — the posterior variance, theoretically optimal

Both work well in practice. The posterior variance tends to give slightly sharper samples.

### Tracing the tensor shapes

- $x_t$ has shape `(B, 1, 28, 28)` for MNIST
- The model output $\epsilon_\theta$ has the same shape
- All schedule values ($\alpha_t$, $\beta_t$, $\bar{\alpha}_t$) are scalars broadcast across the batch

In [ ]:
# --------------------------------------------------------------------------
# Worked Example: DDPM Sampling
# --------------------------------------------------------------------------

@torch.no_grad()
def ddpm_sample(
    model: nn.Module,
    shape: Tuple[int, ...],
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    variance_type: str = "beta",
    track_trajectory: bool = False,
    trajectory_steps: Optional[List[int]] = None,
) -> Tuple[torch.Tensor, List[torch.Tensor]]:
    """DDPM sampling -- Algorithm 2 from Ho et al. 2020.
    
    Args:
        model: Trained noise prediction model.
        shape: (B, C, H, W) shape of samples to generate.
        schedule: Pre-computed schedule dictionary.
        T: Number of diffusion timesteps.
        variance_type: 'beta' for sigma_t^2=beta_t, 'posterior' for tilde_beta_t.
        track_trajectory: If True, save intermediate states.
        trajectory_steps: Which timesteps to save (descending order).
    
    Returns:
        (samples, trajectory) -- final samples and list of intermediate images.
    """
    device = next(model.parameters()).device
    
    # Step 1: Start from pure noise
    x_t = torch.randn(shape, device=device)  # (B, C, H, W)
    
    trajectory = []
    if trajectory_steps is None:
        trajectory_steps = [T - 1, 750, 500, 250, 100, 50, 10, 0]
    
    # Step 2: Reverse process t = T-1, T-2, ..., 0
    for t_val in reversed(range(T)):
        t_batch = torch.full((shape[0],), t_val, device=device, dtype=torch.long)  # (B,)
        
        # Predict noise
        eps_theta = model(x_t, t_batch)  # (B, C, H, W)
        
        # Extract schedule values for this timestep
        beta_t = schedule['betas'][t_val]  # scalar
        alpha_t = schedule['alphas'][t_val]  # scalar
        alpha_bar_t = schedule['alphas_cumprod'][t_val]  # scalar
        sqrt_recip_alpha_t = schedule['sqrt_recip_alphas'][t_val]  # scalar
        sqrt_one_minus_alpha_bar_t = schedule['sqrt_one_minus_alphas_cumprod'][t_val]  # scalar
        
        # Compute posterior mean
        # mu_theta = (1/sqrt(alpha_t)) * (x_t - beta_t/sqrt(1-alpha_bar_t) * eps_theta)
        mu_theta = sqrt_recip_alpha_t * (x_t - beta_t / sqrt_one_minus_alpha_bar_t * eps_theta)  # (B, C, H, W)
        
        # Choose variance
        if variance_type == "beta":
            sigma_t_sq = beta_t  # DDPM default
        elif variance_type == "posterior":
            sigma_t_sq = schedule['posterior_variance'][t_val]  # tilde_beta_t
        else:
            raise ValueError(f"Unknown variance_type: {variance_type}")
        
        # Sample x_{t-1}
        if t_val > 0:
            z = torch.randn_like(x_t)  # (B, C, H, W)
            x_t = mu_theta + torch.sqrt(sigma_t_sq) * z  # (B, C, H, W)
        else:
            x_t = mu_theta  # No noise at final step
        
        # Track trajectory
        if track_trajectory and t_val in trajectory_steps:
            trajectory.append(x_t[0:1].clone())  # save first sample
    
    return x_t, trajectory  # (B, C, H, W), List[(1, C, H, W)]

In [ ]:
# Generate samples and visualize the denoising trajectory
torch.manual_seed(42)

trajectory_steps = [999, 750, 500, 250, 100, 50, 10, 0]

samples, trajectory = ddpm_sample(
    model, shape=(16, 1, 28, 28), schedule=schedule, T=T,
    track_trajectory=True, trajectory_steps=trajectory_steps,
)

print(f"Sample shape: {samples.shape}, range: [{samples.min():.2f}, {samples.max():.2f}]")

# Show denoising trajectory for one sample
show_denoising_trajectory(trajectory, timesteps=trajectory_steps)

# Show full grid of generated samples
show_images(samples, nrow=4, title="DDPM Samples (1000 steps)")

### Exercise 7.1: DDPM Sampling with Both Variance Choices

Let's compare the two variance choices side by side.

**Your task:**

1. Generate 16 samples with `variance_type='beta'`
2. Generate 16 samples with `variance_type='posterior'` (use the **same initial noise**)
3. Display both grids side by side

- Look closely: is there any difference in **sharpness** or **diversity** between the two?
- Hint: use `torch.randn` with a fixed seed to create shared initial noise

In [ ]:
# YOUR CODE HERE — Exercise 7.1

torch.manual_seed(123)
initial_noise = torch.randn(16, 1, 28, 28, device=device)  # (16, 1, 28, 28)

# ===================== YOUR CODE HERE =====================
# 1. Generate samples with variance_type='beta' from initial_noise
#    Hint: you may need to write a helper that accepts x_T as input,
#    or modify the ddpm_sample call
samples_beta = None

# 2. Generate samples with variance_type='posterior' from the same noise
samples_posterior = None
# ====================== END YOUR CODE ======================

# Tests — run this cell to check your work
assert samples_beta is not None, "samples_beta is None — did you generate the samples?"
assert samples_posterior is not None, "samples_posterior is None — did you generate the posterior samples?"
assert samples_beta.shape == (16, 1, 28, 28), f"Expected (16, 1, 28, 28) but got {samples_beta.shape} — did you pass the right batch size?"
assert samples_posterior.shape == (16, 1, 28, 28), f"Expected (16, 1, 28, 28) but got {samples_posterior.shape} — check that you're using the same initial_noise shape"

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, samples, title in [(axes[0], samples_beta, "sigma²=beta"), (axes[1], samples_posterior, "sigma²=posterior")]:
    show_images(samples, nrow=4, title=title)
print("Exercise 7.1 ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# Generate with both variance types, same seed for fair comparison
torch.manual_seed(123)
initial_noise = torch.randn(16, 1, 28, 28, device=device)  # (16, 1, 28, 28)

# We need a version that accepts initial noise
@torch.no_grad()
def ddpm_sample_from_noise(
    model: nn.Module,
    x_T: torch.Tensor,
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    variance_type: str = "beta",
) -> torch.Tensor:
    """DDPM sampling starting from a given noise tensor."""
    device = next(model.parameters()).device
    x_t = x_T.clone()  # (B, C, H, W)

    for t_val in reversed(range(T)):
        t_batch = torch.full((x_t.shape[0],), t_val, device=device, dtype=torch.long)
        eps_theta = model(x_t, t_batch)  # (B, C, H, W)

        beta_t = schedule['betas'][t_val]
        sqrt_recip_alpha_t = schedule['sqrt_recip_alphas'][t_val]
        sqrt_one_minus_alpha_bar_t = schedule['sqrt_one_minus_alphas_cumprod'][t_val]

        mu_theta = sqrt_recip_alpha_t * (x_t - beta_t / sqrt_one_minus_alpha_bar_t * eps_theta)

        sigma_t_sq = schedule['betas'][t_val] if variance_type == 'beta' else schedule['posterior_variance'][t_val]

        if t_val > 0:
            z = torch.randn_like(x_t)
            x_t = mu_theta + torch.sqrt(sigma_t_sq) * z
        else:
            x_t = mu_theta

    return x_t

samples_beta = ddpm_sample_from_noise(model, initial_noise, schedule, T=T, variance_type='beta')
samples_posterior = ddpm_sample_from_noise(model, initial_noise, schedule, T=T, variance_type='posterior')

show_images(samples_beta, nrow=4, title="Variance: beta_t (DDPM default)")
show_images(samples_posterior, nrow=4, title="Variance: posterior (tilde_beta_t)")

# Note: Both produce valid samples. The stochastic noise at each step means
# results diverge even from the same x_T. The posterior variance is theoretically
# tighter (it's the exact posterior when the model is perfect), but in practice
# the difference is often subtle on well-trained models.
print("Both variance choices produce valid samples.")

## DDIM Sampling

DDPM requires stepping through all $T = 1000$ timesteps, which is slow. DDIM (Denoising Diffusion Implicit Models, Song et al. 2020) introduces a family of non-Markovian processes that share the same training objective but allow **skipping timesteps** during sampling.

The key idea: instead of only being able to go from $t$ to $t-1$, DDIM can jump from $t$ to any earlier timestep $t'$.

**The DDIM update rule:**

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \underbrace{\left( \frac{x_t - \sqrt{1 - \bar{\alpha}_t} \cdot \epsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}} \right)}_{\text{predicted } x_0} + \underbrace{\sqrt{1 - \bar{\alpha}_{t-1} - \sigma_t^2} \cdot \epsilon_\theta(x_t, t)}_{\text{direction pointing to } x_t} + \underbrace{\sigma_t \cdot z}_{\text{noise}}$$

The parameter $\eta$ controls stochasticity:

$$\sigma_t = \eta \sqrt{\frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t}} \sqrt{1 - \frac{\bar{\alpha}_t}{\bar{\alpha}_{t-1}}}$$

- $\eta = 1$ recovers DDPM (full stochasticity)
- $\eta = 0$ makes sampling **deterministic** — the same starting noise always produces the same image
- Values in between interpolate smoothly

### Tracing the shapes through sampling

- We first compute the predicted $x_0$ from $x_t$ and $\epsilon_\theta$ — shape stays `(B, 1, 28, 28)`
- Then we combine the predicted $x_0$, the noise direction, and optional stochastic noise to get $x_{t-1}$

In [ ]:
# Worked Example: Stochastic vs Deterministic Sampling

@torch.no_grad()
def sample_with_noise_control(
    model: nn.Module,
    x_T: torch.Tensor,
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    add_noise: bool = True,
) -> torch.Tensor:
    """Sample with or without the stochastic noise term."""
    device = next(model.parameters()).device
    x_t = x_T.clone()

    for t_val in reversed(range(T)):
        t_batch = torch.full((x_t.shape[0],), t_val, device=device, dtype=torch.long)
        eps_theta = model(x_t, t_batch)  # (B, C, H, W)

        beta_t = schedule['betas'][t_val]
        sqrt_recip_alpha_t = schedule['sqrt_recip_alphas'][t_val]
        sqrt_one_minus_alpha_bar_t = schedule['sqrt_one_minus_alphas_cumprod'][t_val]

        mu_theta = sqrt_recip_alpha_t * (x_t - beta_t / sqrt_one_minus_alpha_bar_t * eps_theta)

        if t_val > 0 and add_noise:
            z = torch.randn_like(x_t)
            x_t = mu_theta + torch.sqrt(beta_t) * z
        else:
            x_t = mu_theta

    return x_t

# Same starting noise
torch.manual_seed(42)
shared_noise = torch.randn(4, 1, 28, 28, device=device)  # (4, 1, 28, 28)

# With noise (stochastic) — run 3 times to show diversity
stochastic_runs = []
for i in range(3):
    torch.manual_seed(i * 100)
    result = sample_with_noise_control(model, shared_noise, schedule, T=T, add_noise=True)
    stochastic_runs.append(result)

# Without noise (deterministic) — run 3 times, should be identical
deterministic_runs = []
for i in range(3):
    torch.manual_seed(i * 100)
    result = sample_with_noise_control(model, shared_noise, schedule, T=T, add_noise=False)
    deterministic_runs.append(result)

# Display results
for i in range(3):
    show_images(stochastic_runs[i], nrow=2, title=f"Stochastic run {i+1}")

for i in range(3):
    show_images(deterministic_runs[i], nrow=2, title=f"Deterministic run {i+1}")

# Verify deterministic runs are identical
diff = (deterministic_runs[0] - deterministic_runs[1]).abs().max().item()
print(f"Max pixel difference between deterministic runs: {diff:.6f}")
print("(Should be 0.0 — same x_T with no noise always yields the same x_0)")

### Why DDIM Matters

DDIM isn't just a speed trick — it changes the **nature** of the generative process:

#### Deterministic mapping

With $\eta = 0$, each noise vector maps to exactly one image. This is useful for:
- Interpolation in latent space (blend two noise vectors, get a smooth transition between images)
- Reconstruction (encode an image back to noise, then decode)

#### Fewer steps, same model

DDIM uses the exact same trained model as DDPM. No retraining needed — you just change the sampler at inference time.

#### Speed-quality tradeoff

You can go from 1000 steps (DDPM) to 50 or even 20 steps (DDIM) with only minor quality loss.

The worked example below implements DDIM and demonstrates how $\eta$ controls the balance between determinism and stochasticity.

In [ ]:
# --------------------------------------------------------------------------
# Worked Example: DDIM Sampling
# --------------------------------------------------------------------------

@torch.no_grad()
def ddim_sample(
    model: nn.Module,
    shape: Tuple[int, ...],
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    eta: float = 0.0,
    num_steps: Optional[int] = None,
    track_trajectory: bool = False,
    trajectory_steps: Optional[List[int]] = None,
    x_T: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, List[torch.Tensor]]:
    """DDIM sampling with tunable stochasticity.
    
    Args:
        model: Trained noise prediction model.
        shape: (B, C, H, W) sample shape.
        schedule: Pre-computed schedule dictionary.
        T: Total number of training timesteps.
        eta: Stochasticity parameter. 0=deterministic, 1=DDPM.
        num_steps: Number of sampling steps (None=T for full sampling).
        track_trajectory: Whether to save intermediate states.
        trajectory_steps: Indices into the timestep sequence to save.
        x_T: Optional initial noise (for reproducibility comparisons).
    
    Returns:
        (samples, trajectory)
    """
    device = next(model.parameters()).device
    
    # Create timestep subsequence
    if num_steps is None:
        num_steps = T
    # Uniform spacing: e.g., for 50 steps over T=1000 -> [0, 20, 40, ..., 980]
    step_indices = torch.linspace(0, T - 1, num_steps, dtype=torch.long)  # (num_steps,)
    timesteps = step_indices.flip(0)  # Reverse: high to low
    
    # Start from noise
    if x_T is not None:
        x_t = x_T.clone()
    else:
        x_t = torch.randn(shape, device=device)  # (B, C, H, W)
    
    trajectory = []
    alphas_cumprod = schedule['alphas_cumprod']  # (T,)
    
    for i in range(len(timesteps)):
        t_val = timesteps[i].item()
        t_batch = torch.full((shape[0],), t_val, device=device, dtype=torch.long)  # (B,)
        
        # Predict noise
        eps_theta = model(x_t, t_batch)  # (B, C, H, W)
        
        # Current and previous alpha_bar
        alpha_bar_t = alphas_cumprod[t_val]  # scalar
        if i < len(timesteps) - 1:
            t_prev = timesteps[i + 1].item()
            alpha_bar_t_prev = alphas_cumprod[t_prev]  # scalar
        else:
            alpha_bar_t_prev = torch.tensor(1.0, device=device)  # alpha_bar_0 = 1
        
        # Step 1: Predict x_0
        predicted_x0 = (x_t - torch.sqrt(1.0 - alpha_bar_t) * eps_theta) / torch.sqrt(alpha_bar_t)  # (B, C, H, W)
        predicted_x0 = predicted_x0.clamp(-1.0, 1.0)  # Clamp for stability
        
        # Step 2: Compute sigma_t
        # sigma_t = eta * sqrt((1 - alpha_bar_{t-1}) / (1 - alpha_bar_t)) * sqrt(1 - alpha_bar_t / alpha_bar_{t-1})
        sigma_t = eta * torch.sqrt(
            (1.0 - alpha_bar_t_prev) / (1.0 - alpha_bar_t + 1e-8)
        ) * torch.sqrt(
            1.0 - alpha_bar_t / (alpha_bar_t_prev + 1e-8)
        )
        
        # Step 3: Direction pointing to x_t
        direction = torch.sqrt(
            torch.clamp(1.0 - alpha_bar_t_prev - sigma_t ** 2, min=0.0)
        ) * eps_theta  # (B, C, H, W)
        
        # Step 4: Combine
        x_t = torch.sqrt(alpha_bar_t_prev) * predicted_x0 + direction  # (B, C, H, W)
        
        if sigma_t > 0:
            z = torch.randn_like(x_t)  # (B, C, H, W)
            x_t = x_t + sigma_t * z
        
        # Track trajectory
        if track_trajectory and trajectory_steps is not None and i in trajectory_steps:
            trajectory.append(x_t[0:1].clone())
    
    return x_t, trajectory

In [ ]:
# Demonstrate DDIM with different eta values
torch.manual_seed(42)
shared_noise = torch.randn(8, 1, 28, 28, device=device)  # (8, 1, 28, 28)

eta_values = [0.0, 0.25, 0.5, 0.75, 1.0]

for eta in eta_values:
    samples, _ = ddim_sample(
        model, shape=(8, 1, 28, 28), schedule=schedule, T=T,
        eta=eta, x_T=shared_noise.clone(),
    )
    show_images(samples, nrow=4, title=f"DDIM eta={eta}")

print("Notice: as eta increases, samples become more diverse (more stochastic).")

In [ ]:
# Verify: eta=0 with same x_T always gives same result
torch.manual_seed(42)
fixed_noise = torch.randn(4, 1, 28, 28, device=device)  # (4, 1, 28, 28)

result_a, _ = ddim_sample(model, (4, 1, 28, 28), schedule, T=T, eta=0.0, x_T=fixed_noise.clone())
result_b, _ = ddim_sample(model, (4, 1, 28, 28), schedule, T=T, eta=0.0, x_T=fixed_noise.clone())

max_diff = (result_a - result_b).abs().max().item()
print(f"DDIM eta=0: max difference between two runs from same x_T: {max_diff:.8f}")
print("This confirms deterministic sampling: same noise -> same image.")

### Exercise 7.2: Implement DDIM and Sweep Eta

Now it's your turn — implement DDIM from scratch (without looking at the worked example above), then sweep $\eta$ from 0 to 1.

**Your task:**

1. Implement `ddim_sample_exercise(model, x_T, schedule, T, eta)` that returns final samples
2. Generate 8 samples from the **same initial noise** for each $\eta \in \{0.0, 0.2, 0.4, 0.6, 0.8, 1.0\}$
3. Display the grids

- At $\eta = 0$, same starting noise should always produce the exact same output
- As $\eta$ increases, you should see more variation across runs
- At $\eta = 1$, the behavior matches DDPM

In [ ]:
# YOUR CODE HERE — Exercise 7.2

@torch.no_grad()
def ddim_sample_exercise(
    model: nn.Module,
    x_T: torch.Tensor,
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    eta: float = 0.0,
) -> torch.Tensor:
    """DDIM sampling implementation.
    
    Args:
        model: Trained noise prediction model.
        x_T: Initial noise tensor, shape (B, C, H, W).
        schedule: Pre-computed schedule dict (on device).
        T: Total number of timesteps.
        eta: Stochasticity parameter. 0=deterministic, 1=DDPM.
    
    Returns:
        (B, C, H, W) generated samples.
    """
    device = next(model.parameters()).device
    x_t = x_T.clone()
    alphas_cumprod = schedule['alphas_cumprod']  # (T,)

    # ===================== YOUR CODE HERE =====================
    # Loop from t = T-1 down to 0:
    #   1. Predict noise: eps_theta = model(x_t, t_batch)
    #   2. Get alpha_bar_t and alpha_bar_{t-1} (use 1.0 for t=0)
    #   3. Predict x_0 from x_t and eps_theta
    #   4. Compute sigma_t using eta formula
    #   5. Compute direction term
    #   6. Update: x_t = sqrt(alpha_bar_prev) * pred_x0 + direction + sigma * noise
    pass
    # ====================== END YOUR CODE ======================

    return x_t


# Tests — run this cell to check your work
torch.manual_seed(42)
test_noise = torch.randn(4, 1, 28, 28, device=device)  # (4, 1, 28, 28)
test_result = ddim_sample_exercise(model, test_noise, schedule, T=T, eta=0.0)
assert test_result.shape == (4, 1, 28, 28), f"Expected (4, 1, 28, 28) but got {test_result.shape} — check your loop"
assert torch.isfinite(test_result).all(), "Output contains NaN/Inf — check division by alpha_bar_t"

# Determinism test: eta=0 should give identical results
result_a = ddim_sample_exercise(model, test_noise.clone(), schedule, T=T, eta=0.0)
result_b = ddim_sample_exercise(model, test_noise.clone(), schedule, T=T, eta=0.0)
max_diff = (result_a - result_b).abs().max().item()
assert max_diff < 1e-4, f"eta=0 should be deterministic but max diff is {max_diff:.6f} — are you adding noise when sigma=0?"
print(f"Determinism check: max diff = {max_diff:.8f}")
print("DDIM implementation ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

@torch.no_grad()
def ddim_sample_exercise(
    model: nn.Module,
    x_T: torch.Tensor,
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    eta: float = 0.0,
) -> torch.Tensor:
    """DDIM sampling implementation (exercise version)."""
    device = next(model.parameters()).device
    x_t = x_T.clone()  # (B, C, H, W)
    alphas_cumprod = schedule['alphas_cumprod']  # (T,)

    for t_val in reversed(range(T)):
        t_batch = torch.full((x_t.shape[0],), t_val, device=device, dtype=torch.long)

        # Predict noise
        eps_theta = model(x_t, t_batch)  # (B, C, H, W)

        alpha_bar_t = alphas_cumprod[t_val]
        alpha_bar_prev = alphas_cumprod[t_val - 1] if t_val > 0 else torch.tensor(1.0, device=device)

        # Predict x_0
        pred_x0 = (x_t - torch.sqrt(1.0 - alpha_bar_t) * eps_theta) / torch.sqrt(alpha_bar_t)  # (B, C, H, W)
        pred_x0 = pred_x0.clamp(-1.0, 1.0)

        # Compute sigma
        sigma = eta * torch.sqrt(
            (1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t + 1e-8)
        ) * torch.sqrt(1.0 - alpha_bar_t / (alpha_bar_prev + 1e-8))

        # Direction
        direction = torch.sqrt(torch.clamp(1.0 - alpha_bar_prev - sigma**2, min=0.0)) * eps_theta

        # Update
        x_t = torch.sqrt(alpha_bar_prev) * pred_x0 + direction  # (B, C, H, W)
        if sigma > 0 and t_val > 0:
            x_t = x_t + sigma * torch.randn_like(x_t)

    return x_t

# Sweep eta
torch.manual_seed(42)
fixed_noise = torch.randn(8, 1, 28, 28, device=device)  # (8, 1, 28, 28)

eta_sweep = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
for eta in eta_sweep:
    samples = ddim_sample_exercise(model, fixed_noise.clone(), schedule, T=T, eta=eta)
    show_images(samples, nrow=4, title=f"DDIM eta={eta}")

# Observation: At eta=0 the outputs are identical across runs (deterministic).
# As eta increases, stochastic noise is injected at each step, producing more
# diverse (but noisier) outputs. At eta=1, it is equivalent to DDPM.
print("Eta sweep complete.")

## Accelerated Sampling with Fewer Steps

DDIM lets us skip timesteps, but **which** timesteps should we keep? The choice matters more than you might expect.

### Uniform spacing

Select timesteps evenly across $[0, T]$. Simple and usually good enough.

### Quadratic spacing

Concentrate more steps near $t = 0$ (the end of sampling), where fine details are resolved. The idea is that early steps (high $t$) mostly handle coarse structure, while later steps (low $t$) add textures and edges.

With DDIM, you can often go from 1000 steps down to 50 or even 20 with recognizable output. Let's measure the speed-quality tradeoff.

In [ ]:
# Worked Example: Accelerated Sampling with Fewer Steps

torch.manual_seed(42)
fixed_noise = torch.randn(8, 1, 28, 28, device=device)  # (8, 1, 28, 28)

step_counts = [1000, 200, 50, 20, 10]
timings = {}
results = {}

for n_steps in step_counts:
    start_time = time.time()
    samples, _ = ddim_sample(
        model, shape=(8, 1, 28, 28), schedule=schedule, T=T,
        eta=0.0, num_steps=n_steps, x_T=fixed_noise.clone(),
    )
    elapsed = time.time() - start_time
    timings[n_steps] = elapsed
    results[n_steps] = samples
    print(f"  {n_steps:>5} steps: {elapsed:.2f}s")

# Visual comparison
for n_steps in step_counts:
    show_images(results[n_steps], nrow=4, title=f"DDIM {n_steps} steps ({timings[n_steps]:.1f}s)")

print(f"\nSpeedup from 1000 to 50 steps: {timings[1000] / timings[50]:.1f}x")

In [ ]:
# Plot wall-clock time vs number of steps
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(list(timings.keys()), list(timings.values()), 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Number of Sampling Steps', fontsize=12)
ax.set_ylabel('Wall-Clock Time (seconds)', fontsize=12)
ax.set_title('Sampling Time vs Number of Steps', fontsize=13)
ax.set_xscale('log')
for n, t in timings.items():
    ax.annotate(f"{t:.1f}s", (n, t), textcoords="offset points", xytext=(0, 10), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

### Exercise 7.3: Timestep Sub-Selection for DDIM

When we skip timesteps, the choice of **which** timesteps to keep matters. Let's explore two strategies.

**Your task:**

1. Implement `uniform_timesteps(T, num_steps)` — evenly spaced timestep indices
2. Implement `quadratic_timesteps(T, num_steps)` — more steps concentrated near $t = 0$
3. Compare 50-step DDIM with each strategy visually

- Find the **minimum** number of steps that still produces recognizable MNIST digits
- Does quadratic spacing (more steps near $t = 0$) help with fine details?

In [ ]:
# YOUR CODE HERE — Exercise 7.3

def uniform_timesteps(T: int, num_steps: int) -> torch.Tensor:
    """Uniformly spaced timestep subsequence.
    
    Returns:
        (num_steps,) tensor of timestep indices.
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Return evenly spaced indices from 0 to T-1
    # ====================== END YOUR CODE ======================


def quadratic_timesteps(T: int, num_steps: int) -> torch.Tensor:
    """Quadratic spacing: more steps near t=0 where fine details are resolved.
    
    Returns:
        (num_steps,) tensor of timestep indices.
    """
    # ===================== YOUR CODE HERE =====================
    pass  # Square the linear spacing to concentrate steps near t=0
    # ====================== END YOUR CODE ======================


# Tests — run this cell to check your work
uni = uniform_timesteps(1000, 50)
quad = quadratic_timesteps(1000, 50)

assert uni is not None, "uniform_timesteps returned None"
assert quad is not None, "quadratic_timesteps returned None"
assert uni.shape == (50,), f"Expected (50,) but got {uni.shape} — check your linspace call"
assert quad.shape == (50,), f"Expected (50,) but got {quad.shape}"
assert uni[0] == 0, f"First uniform timestep should be 0, got {uni[0]} — start from 0"
assert uni[-1] == 999, f"Last uniform timestep should be 999, got {uni[-1]} — end at T-1"
# Quadratic should have more steps near t=0
quad_near_zero = (quad < 100).sum().item()
uni_near_zero = (uni < 100).sum().item()
assert quad_near_zero > uni_near_zero, f"Quadratic has {quad_near_zero} steps near t=0 vs uniform {uni_near_zero} — quadratic should concentrate steps near 0"
print(f"Uniform:   first 5 = {uni[:5].tolist()}, last 5 = {uni[-5:].tolist()}")
print(f"Quadratic: first 5 = {quad[:5].tolist()}, last 5 = {quad[-5:].tolist()}")
print("Timestep sub-selection ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def uniform_timesteps(T: int, num_steps: int) -> torch.Tensor:
    """Uniformly spaced timestep subsequence."""
    return torch.linspace(0, T - 1, num_steps, dtype=torch.long)


def quadratic_timesteps(T: int, num_steps: int) -> torch.Tensor:
    """Quadratic spacing: more steps near t=0 where fine details are resolved."""
    t = torch.linspace(0, 1, num_steps) ** 2  # quadratic in [0, 1]
    return (t * (T - 1)).long()


@torch.no_grad()
def ddim_sample_custom_steps(
    model: nn.Module,
    x_T: torch.Tensor,
    schedule: Dict[str, torch.Tensor],
    timestep_seq: torch.Tensor,
    eta: float = 0.0,
) -> torch.Tensor:
    """DDIM sampling with a custom timestep sequence."""
    device = next(model.parameters()).device
    x_t = x_T.clone()
    alphas_cumprod = schedule['alphas_cumprod']

    # Reverse the sequence (high to low)
    timesteps = timestep_seq.flip(0)

    for i in range(len(timesteps)):
        t_val = timesteps[i].item()
        t_batch = torch.full((x_t.shape[0],), t_val, device=device, dtype=torch.long)

        eps_theta = model(x_t, t_batch)  # (B, C, H, W)

        alpha_bar_t = alphas_cumprod[t_val]
        if i < len(timesteps) - 1:
            alpha_bar_prev = alphas_cumprod[timesteps[i + 1].item()]
        else:
            alpha_bar_prev = torch.tensor(1.0, device=device)

        pred_x0 = ((x_t - torch.sqrt(1.0 - alpha_bar_t) * eps_theta) / torch.sqrt(alpha_bar_t)).clamp(-1, 1)

        sigma = eta * torch.sqrt(
            (1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t + 1e-8)
        ) * torch.sqrt(1.0 - alpha_bar_t / (alpha_bar_prev + 1e-8))

        direction = torch.sqrt(torch.clamp(1.0 - alpha_bar_prev - sigma**2, min=0.0)) * eps_theta
        x_t = torch.sqrt(alpha_bar_prev) * pred_x0 + direction

        if sigma > 0:
            x_t = x_t + sigma * torch.randn_like(x_t)

    return x_t

# Compare uniform vs quadratic at 50 steps
torch.manual_seed(42)
fixed_noise = torch.randn(8, 1, 28, 28, device=device)

uniform_seq = uniform_timesteps(T, 50)
quadratic_seq = quadratic_timesteps(T, 50)

samples_uniform = ddim_sample_custom_steps(model, fixed_noise.clone(), schedule, uniform_seq, eta=0.0)
samples_quadratic = ddim_sample_custom_steps(model, fixed_noise.clone(), schedule, quadratic_seq, eta=0.0)

show_images(samples_uniform, nrow=4, title="Uniform spacing (50 steps)")
show_images(samples_quadratic, nrow=4, title="Quadratic spacing (50 steps)")

# Find minimum steps
print("\nMinimum steps search:")
for n in [5, 10, 20, 50]:
    seq = uniform_timesteps(T, n)
    s = ddim_sample_custom_steps(model, fixed_noise[:4].clone(), schedule, seq, eta=0.0)
    show_images(s, nrow=2, title=f"DDIM {n} steps")
    print(f"  {n:>3} steps — range: [{s.min():.2f}, {s.max():.2f}]")

### Visualizing the Noise Schedule at Different Step Counts

When we sub-select timesteps, we're choosing which $\bar{\alpha}_t$ values to visit during sampling. Let's see how the schedule coverage changes as we reduce the number of steps.

Notice that with very few steps, large gaps appear in the schedule — the sampler has to make bigger jumps, which can degrade quality.

In [ ]:
# Visualize which alpha_bar values are used at different step counts
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

alphas_cumprod_np = schedule['alphas_cumprod'].cpu().numpy()

# Left: Full schedule with sub-selected points
axes[0].plot(alphas_cumprod_np, color='lightgray', linewidth=1, label='Full schedule')
for n_steps, color in [(50, 'steelblue'), (20, 'darkorange'), (10, 'crimson')]:
    indices = torch.linspace(0, T - 1, n_steps, dtype=torch.long).numpy()
    axes[0].scatter(indices, alphas_cumprod_np[indices], s=15, color=color, label=f'{n_steps} steps', zorder=5)
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('alpha_bar_t')
axes[0].set_title('Sub-selected Schedule Points')
axes[0].legend(fontsize=9)

# Right: Uniform vs quadratic spacing
n = 20
uni = uniform_timesteps(T, n).numpy()
quad = quadratic_timesteps(T, n).numpy()
axes[1].stem(uni, alphas_cumprod_np[uni], linefmt='steelblue', markerfmt='o', basefmt=' ', label='Uniform')
axes[1].stem(quad, alphas_cumprod_np[quad], linefmt='darkorange', markerfmt='s', basefmt=' ', label='Quadratic')
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel('alpha_bar_t')
axes[1].set_title(f'Spacing Strategies ({n} steps)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Sampling Pipeline

In practice, you want a single function that handles all the details — choosing DDPM vs DDIM, setting the number of steps and $\eta$, managing the noise schedule, and optionally recording intermediate states.

Let's wrap everything into a clean `sampling_pipeline` function that you can reuse going forward.

In [ ]:
# --------------------------------------------------------------------------
# Worked Example: Full End-to-End Sampling Pipeline
# --------------------------------------------------------------------------

def dynamic_threshold(x0_pred: torch.Tensor, percentile: float = 0.995) -> torch.Tensor:
    """Dynamic thresholding from Imagen (Saharia et al. 2022).
    
    Instead of hard-clamping to [-1, 1], compute a data-dependent threshold
    and rescale. This prevents saturation in high-guidance settings.
    
    Args:
        x0_pred: (B, C, H, W) predicted clean image.
        percentile: Percentile for computing the threshold (default 99.5%).
    Returns:
        Thresholded tensor in [-1, 1].
    """
    B = x0_pred.shape[0]
    # Compute per-sample threshold
    flat = x0_pred.reshape(B, -1).abs()  # (B, C*H*W)
    threshold = torch.quantile(flat, percentile, dim=1, keepdim=True)  # (B, 1)
    threshold = torch.clamp(threshold, min=1.0)  # At least 1.0
    # Reshape for broadcasting
    threshold = threshold[:, :, None, None]  # (B, 1, 1, 1)
    # Clamp and rescale
    return torch.clamp(x0_pred, -threshold, threshold) / threshold  # (B, C, H, W)


@torch.no_grad()
def sampling_pipeline(
    model: nn.Module,
    num_samples: int,
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    method: str = "ddim",
    num_steps: Optional[int] = None,
    eta: float = 0.0,
    use_dynamic_threshold: bool = False,
    clamp_each_step: bool = True,
    seed: Optional[int] = None,
    image_shape: Tuple[int, ...] = (1, 28, 28),
) -> torch.Tensor:
    """Complete sampling pipeline with all practical details.
    
    Args:
        model: Trained noise prediction model.
        num_samples: Number of images to generate.
        schedule: Pre-computed schedule dictionary.
        T: Total training timesteps.
        method: 'ddpm' or 'ddim'.
        num_steps: Sampling steps (None=T).
        eta: DDIM stochasticity (ignored for DDPM).
        use_dynamic_threshold: Apply dynamic thresholding to x0 predictions.
        clamp_each_step: Clamp x_t to [-1, 1] at each step.
        seed: Random seed for reproducibility.
        image_shape: (C, H, W) shape of each image.
    
    Returns:
        (num_samples, C, H, W) tensor in [0, 1] ready for display.
    """
    device = next(model.parameters()).device
    if seed is not None:
        torch.manual_seed(seed)
    
    shape = (num_samples, *image_shape)  # (B, C, H, W)
    
    if method == "ddpm":
        samples, _ = ddpm_sample(model, shape, schedule, T=T)
    elif method == "ddim":
        if num_steps is None:
            num_steps = T
        samples, _ = ddim_sample(model, shape, schedule, T=T, eta=eta, num_steps=num_steps)
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Final clamp and denormalize
    if use_dynamic_threshold:
        samples = dynamic_threshold(samples)
    else:
        samples = samples.clamp(-1.0, 1.0)  # (B, C, H, W)
    
    samples = denormalize(samples)  # (B, C, H, W) in [0, 1]
    return samples


# Generate a 4x4 grid
samples = sampling_pipeline(
    model, num_samples=16, schedule=schedule, T=T,
    method='ddim', num_steps=50, eta=0.0, seed=42,
)
show_images(samples, nrow=4, title="Pipeline Output: 4x4 Grid (DDIM, 50 steps)")

In [ ]:
# Show the full denoising process for one sample as a row of images
torch.manual_seed(42)
trajectory_indices = list(range(0, 50, 5)) + [49]  # Every 5th step + final

_, trajectory = ddim_sample(
    model, shape=(1, 1, 28, 28), schedule=schedule, T=T,
    eta=0.0, num_steps=50,
    track_trajectory=True, trajectory_steps=trajectory_indices,
)

# Map trajectory indices to actual timestep values for display
step_indices = torch.linspace(0, T - 1, 50, dtype=torch.long).flip(0)
display_timesteps = [step_indices[i].item() for i in trajectory_indices if i < len(step_indices)]

if trajectory:
    show_denoising_trajectory(
        trajectory,
        timesteps=display_timesteps[:len(trajectory)],
    )

### Exercise 7.4: Full Pipeline with DDPM and DDIM

Let's put the pipeline through its paces.

**Your task:**

1. Generate 16 samples with **DDPM** (1000 steps)
2. Generate 16 samples with **DDIM** (50 steps, $\eta = 0$)
3. Measure wall-clock time for each
4. Display side by side

- Can you tell which is which visually?
- How much faster is DDIM?

In [ ]:
# YOUR CODE HERE — Exercise 7.4

# ===================== YOUR CODE HERE =====================
# 1. Time DDPM: sampling_pipeline(model, 16, schedule, T=T, method='ddpm', seed=42)
samples_ddpm = None
time_ddpm = 0.0

# 2. Time DDIM: sampling_pipeline(model, 16, schedule, T=T, method='ddim', num_steps=50, eta=0.0, seed=42)
samples_ddim = None
time_ddim = 0.0
# ====================== END YOUR CODE ======================

# Tests — run this cell to check your work
assert samples_ddpm is not None, "samples_ddpm is None — did you call sampling_pipeline?"
assert samples_ddim is not None, "samples_ddim is None — did you call sampling_pipeline?"
assert samples_ddpm.shape == (16, 1, 28, 28), f"DDPM shape: {samples_ddpm.shape} — did you pass num_samples=16?"
assert samples_ddim.shape == (16, 1, 28, 28), f"DDIM shape: {samples_ddim.shape} — did you pass num_samples=16?"
assert time_ddpm > 0, "time_ddpm is 0 — wrap your pipeline call in time.time() measurements"
assert time_ddim > 0, "time_ddim is 0 — wrap your pipeline call in time.time() measurements"

show_images(samples_ddpm, nrow=4, title=f"DDPM (1000 steps, {time_ddpm:.1f}s)")
show_images(samples_ddim, nrow=4, title=f"DDIM (50 steps, {time_ddim:.1f}s)")
print(f"Speedup: {time_ddpm / time_ddim:.1f}x")
print("Pipeline comparison ✓")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# DDPM (full 1000 steps)
start = time.time()
samples_ddpm = sampling_pipeline(
    model, num_samples=16, schedule=schedule, T=T,
    method='ddpm', seed=42,
)
time_ddpm = time.time() - start

# DDIM (50 steps, deterministic)
start = time.time()
samples_ddim = sampling_pipeline(
    model, num_samples=16, schedule=schedule, T=T,
    method='ddim', num_steps=50, eta=0.0, seed=42,
)
time_ddim = time.time() - start

show_images(samples_ddpm, nrow=4, title=f"DDPM (1000 steps, {time_ddpm:.1f}s)")
show_images(samples_ddim, nrow=4, title=f"DDIM (50 steps, {time_ddim:.1f}s)")

print(f"DDPM: {time_ddpm:.2f}s")
print(f"DDIM: {time_ddim:.2f}s")
print(f"Speedup: {time_ddpm / time_ddim:.1f}x")

## Probability Flow ODE

Here's where things get really elegant. There's a deep connection between diffusion models and ordinary differential equations.

DDIM with $\eta = 0$ is actually solving a discretized ODE called the **probability flow ODE**:

$$\frac{dx}{dt} = f(x, t) - \frac{1}{2} g(t)^2 \nabla_x \log p_t(x)$$

where the score $\nabla_x \log p_t(x)$ is estimated by our noise-prediction network.

In practice, we can write this as a simple Euler integration:

$$x_{t - \Delta t} = x_t + \Delta t \cdot \text{velocity}(x_t, t)$$

The key insight: **DDIM ($\eta = 0$) and the Euler ODE solver produce identical results** — they are the same algorithm written differently. This connection opens the door to more advanced ODE solvers (Runge-Kutta, adaptive step sizes) for even faster sampling.

In [ ]:
# --------------------------------------------------------------------------
# Worked Example: Probability Flow ODE via Manual Euler Method
# --------------------------------------------------------------------------

@torch.no_grad()
def probability_flow_ode_euler(
    model: nn.Module,
    shape: Tuple[int, ...],
    schedule: Dict[str, torch.Tensor],
    T: int = 1000,
    num_steps: int = 1000,
    x_T: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Sample via Euler discretization of the probability flow ODE.
    
    The ODE in terms of alpha_bar:
        dx/d(alpha_bar) = (x - eps_theta) / (2 * alpha_bar)
                        - eps_theta / (2 * (1 - alpha_bar))
    
    We discretize this using the DDIM(eta=0) update, which IS the Euler method
    for this ODE. Here we implement it explicitly to show the connection.
    
    Args:
        model: Trained noise prediction model.
        shape: (B, C, H, W) sample shape.
        schedule: Pre-computed schedule dict.
        T: Total training timesteps.
        num_steps: Number of Euler steps.
        x_T: Optional initial noise.
    
    Returns:
        (B, C, H, W) generated samples.
    """
    device = next(model.parameters()).device
    
    if x_T is not None:
        x_t = x_T.clone()
    else:
        x_t = torch.randn(shape, device=device)  # (B, C, H, W)
    
    # Create timestep sequence
    step_indices = torch.linspace(0, T - 1, num_steps, dtype=torch.long)
    timesteps = step_indices.flip(0)  # high to low
    
    alphas_cumprod = schedule['alphas_cumprod']
    
    for i in range(len(timesteps)):
        t_val = timesteps[i].item()
        t_batch = torch.full((shape[0],), t_val, device=device, dtype=torch.long)
        
        # Get score estimate from noise prediction
        eps_theta = model(x_t, t_batch)  # (B, C, H, W)
        
        alpha_bar_t = alphas_cumprod[t_val]
        if i < len(timesteps) - 1:
            alpha_bar_prev = alphas_cumprod[timesteps[i + 1].item()]
        else:
            alpha_bar_prev = torch.tensor(1.0, device=device)
        
        # DDIM(eta=0) update = Euler step of the probability flow ODE
        pred_x0 = ((x_t - torch.sqrt(1 - alpha_bar_t) * eps_theta) / torch.sqrt(alpha_bar_t)).clamp(-1, 1)
        direction = torch.sqrt(1 - alpha_bar_prev) * eps_theta
        x_t = torch.sqrt(alpha_bar_prev) * pred_x0 + direction  # (B, C, H, W)
    
    return x_t


# Compare ODE Euler to DDIM(eta=0) -- they should match
torch.manual_seed(42)
fixed_noise = torch.randn(4, 1, 28, 28, device=device)  # (4, 1, 28, 28)

samples_ode = probability_flow_ode_euler(
    model, (4, 1, 28, 28), schedule, T=T, num_steps=50, x_T=fixed_noise.clone(),
)
samples_ddim_ref, _ = ddim_sample(
    model, (4, 1, 28, 28), schedule, T=T, eta=0.0, num_steps=50, x_T=fixed_noise.clone(),
)

max_diff = (samples_ode - samples_ddim_ref).abs().max().item()
print(f"Max difference between ODE Euler and DDIM(eta=0): {max_diff:.8f}")
print("These are the same algorithm -- DDIM(eta=0) IS the Euler method for the probability flow ODE.")

print("\nProbability Flow ODE (Euler):")
show_images(samples_ode, nrow=2, title="Probability Flow ODE (Euler)")

print("\nDDIM (eta=0):")
show_images(samples_ddim_ref, nrow=2, title="DDIM (eta=0)")

print("If these look identical, that confirms: DDIM(eta=0) IS the probability flow ODE.")

## Capstone: Putting It All Together

This is where everything comes together. You'll run a comprehensive comparison of all the sampling methods we've covered:

1. Generate 64 samples with both **DDPM** (1000 steps) and **DDIM** (50 steps, $\eta = 0$), timing both
2. Compare DDIM at **different step counts** (1000, 100, 50, 20, 10) to see the quality/speed tradeoff
3. Visualize **denoising trajectories** for both DDPM and DDIM
4. Show that DDIM is **deterministic** (same noise → same image) while DDPM is **stochastic**
5. Build a **summary table** comparing methods, step counts, and timings

This capstone is large — the solution is broken into parts. Try as much as you can, then check the solutions for any parts you got stuck on.

In [ ]:
# YOUR CODE HERE — Capstone

# ===================== YOUR CODE HERE =====================
# Part 1: Generate 64 samples with DDPM and DDIM, time both
#   - Use sampling_pipeline with method='ddpm' and method='ddim'
#   - Display both grids and print the speedup

# Part 2: DDIM at different step counts
#   - Try step_counts = [1000, 100, 50, 20, 10]
#   - Time each, display the grids

# Part 3: Denoising trajectories
#   - Show DDPM trajectory at timesteps [999, 750, 500, 250, 100, 50, 10, 0]
#   - Show DDIM (50 steps) trajectory at evenly spaced indices

# Part 4: Determinism test
#   - Create shared_x_T = torch.randn(4, 1, 28, 28, device=device)
#   - Run DDIM (eta=0) 3 times from the same x_T — outputs should be identical
#   - Run DDPM 3 times from the same x_T — outputs should differ

# Part 5: Summary table
#   - Print a table comparing DDPM, DDIM at various step counts
#   - Include columns: method, steps, time, deterministic?

# ====================== END YOUR CODE ======================
print("Complete the capstone above, then check the solution cells below.")

In [ ]:
# ✅ SOLUTION — Part 1: Generate 64 samples with DDPM and DDIM

print("=" * 60)
print("CAPSTONE: Comprehensive DDPM vs DDIM Comparison")
print("=" * 60)

# DDPM: 64 samples, full 1000 steps
start = time.time()
samples_ddpm_64 = sampling_pipeline(
    model, num_samples=64, schedule=schedule, T=T,
    method='ddpm', seed=42,
)
time_ddpm_64 = time.time() - start
print(f"\nDDPM (1000 steps): {time_ddpm_64:.1f}s for 64 samples")

# DDIM: 64 samples, 50 steps, deterministic
start = time.time()
samples_ddim_64 = sampling_pipeline(
    model, num_samples=64, schedule=schedule, T=T,
    method='ddim', num_steps=50, eta=0.0, seed=42,
)
time_ddim_64 = time.time() - start
print(f"DDIM (50 steps):   {time_ddim_64:.1f}s for 64 samples")
print(f"Speedup:           {time_ddpm_64 / time_ddim_64:.1f}x")

# Display both grids
show_images(samples_ddpm_64, nrow=8, title="DDPM Samples (64, 1000 steps)")
show_images(samples_ddim_64, nrow=8, title="DDIM Samples (64, 50 steps, eta=0)")

In [ ]:
# ✅ SOLUTION — Part 2: DDIM at different step counts

print("\n" + "=" * 60)
print("DDIM: Step Count Comparison")
print("=" * 60)

ddim_step_counts = [1000, 100, 50, 20, 10]
ddim_samples = {}
ddim_timings = {}

for n_steps in ddim_step_counts:
    start = time.time()
    samples = sampling_pipeline(
        model, num_samples=16, schedule=schedule, T=T,
        method='ddim', num_steps=n_steps, eta=0.0, seed=42,
    )
    elapsed = time.time() - start
    ddim_samples[n_steps] = samples
    ddim_timings[n_steps] = elapsed
    print(f"  {n_steps:>5} steps: {elapsed:.2f}s")

for n_steps in ddim_step_counts:
    show_images(ddim_samples[n_steps], nrow=4, title=f"DDIM {n_steps} steps ({ddim_timings[n_steps]:.1f}s)")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this — Part 3: Denoising trajectories

print("\n" + "=" * 60)
print("Denoising Trajectories")
print("=" * 60)

# DDPM trajectory
torch.manual_seed(42)
traj_steps_ddpm = [999, 750, 500, 250, 100, 50, 10, 0]
_, traj_ddpm = ddpm_sample(
    model, shape=(1, 1, 28, 28), schedule=schedule, T=T,
    track_trajectory=True, trajectory_steps=traj_steps_ddpm,
)
print(f"DDPM trajectory: {len(traj_ddpm)} snapshots")
show_denoising_trajectory(traj_ddpm, timesteps=traj_steps_ddpm[:len(traj_ddpm)])

# DDIM trajectory (50 steps)
torch.manual_seed(42)
traj_indices_ddim = [0, 5, 10, 15, 20, 30, 40, 49]
_, traj_ddim = ddim_sample(
    model, shape=(1, 1, 28, 28), schedule=schedule, T=T,
    eta=0.0, num_steps=50,
    track_trajectory=True, trajectory_steps=traj_indices_ddim,
)

step_indices = torch.linspace(0, T - 1, 50, dtype=torch.long).flip(0)
display_ts = [step_indices[i].item() for i in traj_indices_ddim if i < len(step_indices)]
print(f"DDIM trajectory: {len(traj_ddim)} snapshots")
if traj_ddim:
    show_denoising_trajectory(traj_ddim, timesteps=display_ts[:len(traj_ddim)])

In [ ]:
# ✅ SOLUTION — Part 4: Deterministic DDIM vs Stochastic DDPM

print("\n" + "=" * 60)
print("Determinism Test")
print("=" * 60)

torch.manual_seed(42)
shared_x_T = torch.randn(4, 1, 28, 28, device=device)  # (4, 1, 28, 28)

# DDIM (eta=0): Same seed -> same image (deterministic)
ddim_run1, _ = ddim_sample(model, (4, 1, 28, 28), schedule, T=T, eta=0.0, num_steps=50, x_T=shared_x_T.clone())
ddim_run2, _ = ddim_sample(model, (4, 1, 28, 28), schedule, T=T, eta=0.0, num_steps=50, x_T=shared_x_T.clone())
ddim_run3, _ = ddim_sample(model, (4, 1, 28, 28), schedule, T=T, eta=0.0, num_steps=50, x_T=shared_x_T.clone())

print(f"DDIM runs max diff (run1 vs run2): {(ddim_run1 - ddim_run2).abs().max().item():.8f}")
print(f"DDIM runs max diff (run1 vs run3): {(ddim_run1 - ddim_run3).abs().max().item():.8f}")
print("-> Deterministic: identical outputs from same x_T.")

# DDPM: Same x_T -> different images (stochastic per-step noise)
ddpm_run1 = ddpm_sample_from_noise(model, shared_x_T.clone(), schedule, T=T)
ddpm_run2 = ddpm_sample_from_noise(model, shared_x_T.clone(), schedule, T=T)
ddpm_run3 = ddpm_sample_from_noise(model, shared_x_T.clone(), schedule, T=T)

print(f"\nDDPM runs max diff (run1 vs run2): {(ddpm_run1 - ddpm_run2).abs().max().item():.4f}")
print(f"DDPM runs max diff (run1 vs run3): {(ddpm_run1 - ddpm_run3).abs().max().item():.4f}")
print("-> Stochastic: different outputs even from same x_T.")

# Visual comparison
show_images(torch.cat([ddim_run1, ddim_run2, ddim_run3], dim=0), nrow=4, title="DDIM (eta=0): 3 runs from same x_T — identical")
show_images(torch.cat([ddpm_run1, ddpm_run2, ddpm_run3], dim=0), nrow=4, title="DDPM: 3 runs from same x_T — different")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this — Part 5: Summary table

print("\n" + "=" * 60)
print("Summary")
print("=" * 60)

print(f"\n{'Method':<18} | {'Steps':>5} | {'Time (s)':>9} | {'Deterministic':>14}")
print("-" * 55)
print(f"{'DDPM':<18} | {'1000':>5} | {time_ddpm_64:>9.1f} | {'No':>14}")
for n_steps in ddim_step_counts:
    det = 'Yes' if True else 'No'
    print(f"{'DDIM (eta=0)':<18} | {n_steps:>5} | {ddim_timings[n_steps]:>9.2f} | {det:>14}")

print("\nKey Takeaways:")
print("- DDPM requires 1000 steps and is stochastic (diverse but slow).")
print("- DDIM(eta=0) is deterministic: same noise -> same image, enabling interpolation.")
print("- DDIM with 50 steps gives near-DDPM quality at ~20x speedup.")
print("- Both sample from the same learned distribution p_theta(x_0).")
print("- The probability flow ODE perspective unifies these approaches.")